# MãosLivres — Análise dos resultados

**CitizenTech | Global Solution FIAP 2026.2**  
Disciplina: Cognitive Computing, Computer Vision and IoT Systems

Este notebook reúne as duas avaliações do projeto:

1. **Desempenho computacional** — o sistema roda em tempo real? (dados do `benchmark_desempenho.py`)
2. **Desempenho de uso** — a pessoa consegue clicar com precisão? (dados do `teste_validacao.py`)

## Como usar

Antes de executar, gere os dados:

```powershell
venv\Scripts\python.exe benchmark_desempenho.py   # sente-se na frente da câmera
.\iniciar.bat                                      # numa janela
.\validar.bat                                      # em outra, com 3 a 5 pessoas
```

Depois use **Run All**. Se um arquivo de dados não existir, a célula avisa em vez de falhar.

In [ ]:
# Bibliotecas. Nenhuma dependência nova: matplotlib já está no requirements.txt.
import csv
from pathlib import Path
from statistics import mean, median, pstdev

import matplotlib.pyplot as plt

PASTA = Path("resultados")
CSV_QUADROS = PASTA / "desempenho_quadros.csv"
CSV_VALIDACAO = Path("resultados_validacao.csv")

print("benchmark disponível: ", CSV_QUADROS.exists())
print("validação disponível:", CSV_VALIDACAO.exists())

---
## 1. Desempenho computacional

**Pergunta:** o processamento de visão computacional cabe no orçamento de tempo real?

Uma webcam de 30 FPS entrega um quadro a cada **33,3 ms**. Se o nosso processamento levar mais que isso, o cursor atrasa e o sistema fica inutilizável. A métrica que importa é o tempo do **nosso processamento** (pré-processamento + inferência + pós-processamento), separado da espera pela câmera.

In [ ]:
if not CSV_QUADROS.exists():
    print("Rode primeiro: venv\\Scripts\\python.exe benchmark_desempenho.py")
    quadros = []
else:
    with open(CSV_QUADROS, encoding="utf-8-sig") as f:
        quadros = [{k: float(v) for k, v in linha.items()} for linha in csv.DictReader(f)]
    print(f"{len(quadros)} quadros carregados")

In [ ]:
if quadros:
    proc = [q["processamento_ms"] for q in quadros]
    infer = [q["inferencia_ms"] for q in quadros]
    espera = [q["captura_ms"] for q in quadros]
    fps = [q["fps_ciclo"] for q in quadros]
    deteccao = 100 * sum(q["rosto_detectado"] for q in quadros) / len(quadros)

    def p95(v):
        return sorted(v)[int(0.95 * (len(v) - 1))]

    print(f"{'métrica':<34}{'média':>10}{'mediana':>10}{'p95':>10}")
    for nome, v in (("inferência MediaPipe (ms)", infer),
                    ("processamento total (ms)", proc),
                    ("espera da webcam (ms)", espera),
                    ("FPS real", fps)):
        print(f"{nome:<34}{mean(v):>10.2f}{median(v):>10.2f}{p95(v):>10.2f}")

    print(f"\nTaxa de detecção do rosto: {deteccao:.1f}%")
    ocupacao = 100 * mean(proc) / 33.3
    print(f"Ocupação do orçamento de 33,3 ms: {ocupacao:.0f}%")
    print(f"Folga computacional: {33.3 / mean(proc):.0f}x")
    if deteccao < 50:
        print("\nATENÇÃO: detecção baixa. Refaça o benchmark sentada na frente da câmera.")

In [ ]:
if quadros:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    ax1.hist(proc, bins=30, color="#2d7dd2", edgecolor="white")
    ax1.axvline(mean(proc), color="#d62828", linestyle="--", label=f"média {mean(proc):.1f} ms")
    ax1.axvline(33.3, color="#2a9d8f", linestyle=":", label="orçamento 33,3 ms")
    ax1.set_xlabel("Processamento por quadro (ms)")
    ax1.set_ylabel("Quadros")
    ax1.set_title("Latência do processamento")
    ax1.legend()

    etapas = ["pré-proc.", "inferência", "pós-proc.", "espera webcam"]
    valores = [mean([q["preproc_ms"] for q in quadros]), mean(infer),
               mean([q["posproc_ms"] for q in quadros]), mean(espera)]
    ax2.barh(etapas, valores, color=["#9bc1bc", "#e76f51", "#8e7dbe", "#adb5bd"])
    for i, v in enumerate(valores):
        ax2.text(v, i, f"  {v:.1f} ms", va="center")
    ax2.set_xlim(0, max(valores) * 1.3)
    ax2.set_xlabel("Tempo médio (ms)")
    ax2.set_title("Onde o tempo é gasto")

    plt.tight_layout()
    plt.show()

### Leitura crítica do desempenho

O achado relevante é que **o gargalo não é o algoritmo, é a webcam**. A inferência do MediaPipe custa poucos milissegundos, enquanto a maior parte do ciclo é o `cap.read()` esperando o próximo quadro chegar do hardware.

Consequências para o projeto:

- Sobra folga de CPU, então o sistema roda em notebook comum **sem placa de vídeo dedicada**, o que é essencial para uma solução de acessibilidade de custo zero.
- Otimizar o algoritmo não aumentaria o FPS. Para ganhar fluidez, o caminho seria uma câmera melhor ou capturar em outra *thread*.
- A folga disponível é o que viabiliza os próximos passos (teclado próprio, mais gestos) sem comprometer o tempo real.

---
## 2. Desempenho de uso (validação com participantes)

**Pergunta:** a pessoa consegue acertar alvos usando só o rosto, e a que custo de tempo?

Protocolo: 8 alvos por rodada, posições sorteadas com no mínimo 200 px de distância do alvo anterior. Cada participante faz uma rodada no modo **rosto** e uma no modo **mouse**. O mouse é **referência de comparação**, não meta.

In [ ]:
if not CSV_VALIDACAO.exists():
    print("Rode o validar.bat com 3 a 5 pessoas (modo rosto e modo mouse).")
    print("NÃO invente números: o relatório precisa dos dados reais.")
    dados = []
else:
    with open(CSV_VALIDACAO, encoding="utf-8-sig") as f:
        dados = [{
            "participante": l["participante"],
            "modo": l["modo"].strip().lower(),
            "acertos": int(l["acertos"]),
            "erros": int(l["erros"]),
            "taxa": float(l["taxa_acerto_%"]),
            "tempo": float(l["tempo_medio_s"]),
        } for l in csv.DictReader(f)]
    print(f"{len(dados)} rodadas | participantes: {sorted({d['participante'] for d in dados})}")

In [ ]:
if dados:
    resumo = {}
    for modo in ("rosto", "mouse"):
        g = [d for d in dados if d["modo"] == modo]
        if g:
            resumo[modo] = {
                "n": len({d["participante"] for d in g}),
                "taxa": mean(d["taxa"] for d in g),
                "taxa_sd": pstdev([d["taxa"] for d in g]) if len(g) > 1 else 0.0,
                "tempo": mean(d["tempo"] for d in g),
                "tempo_sd": pstdev([d["tempo"] for d in g]) if len(g) > 1 else 0.0,
            }

    print(f"{'modo':<8}{'N':>4}{'acerto':>12}{'tempo/alvo':>14}")
    for modo, r in resumo.items():
        print(f"{modo:<8}{r['n']:>4}{r['taxa']:>10.1f}%{r['tempo']:>12.2f}s")

    if "rosto" in resumo and "mouse" in resumo:
        print(f"\nCusto relativo de tempo: {resumo['rosto']['tempo']/resumo['mouse']['tempo']:.1f}x o mouse")
        print(f"Diferença de acerto: {resumo['rosto']['taxa']-resumo['mouse']['taxa']:+.1f} p.p.")
    if max((r["n"] for r in resumo.values()), default=0) < 3:
        print("\nAVISO: menos de 3 participantes. Trate como estudo exploratório.")

In [ ]:
if dados and resumo:
    modos = list(resumo.keys())
    cores = {"rosto": "#2d7dd2", "mouse": "#adb5bd"}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    for ax, campo, sd, titulo, rotulo, fmt in (
            (ax1, "taxa", "taxa_sd", "Taxa de acerto por modo", "Acerto (%)", "{:.1f}%"),
            (ax2, "tempo", "tempo_sd", "Tempo médio por alvo", "Segundos", "{:.2f}s")):
        vals = [resumo[m][campo] for m in modos]
        errs = [resumo[m][sd] for m in modos]
        barras = ax.bar(modos, vals, yerr=errs, capsize=8, width=0.55,
                        color=[cores.get(m, "#888") for m in modos])
        for b, v in zip(barras, vals):
            ax.text(b.get_x() + b.get_width()/2, b.get_height(), fmt.format(v),
                    ha="center", va="bottom", fontweight="bold")
        ax.set_ylabel(rotulo)
        ax.set_title(titulo)
        ax.set_ylim(0, max(v + e for v, e in zip(vals, errs)) * 1.25)

    plt.tight_layout()
    plt.show()

---
## 3. Considerações finais

**O que foi comprovado.** O processamento de visão computacional roda com folga em tempo real num notebook comum, e o controle do cursor por gestos do rosto permite executar as operações básicas de interação: mover, clicar, rolar, arrastar e pausar.

**Limitações desta avaliação.** Os participantes não são pessoas sem braços, então os números medem **viabilidade técnica**, não o desempenho do público-alvo. A amostra é pequena e não permite inferência estatística. Nenhuma das duas coisas invalida o resultado, mas as duas precisam estar declaradas.

**Limitação funcional.** O projeto não digita texto. Investigamos sete abordagens e documentamos a causa: o Windows descarta cliques gerados por programa em janelas privilegiadas como o teclado virtual (`osk.exe`). A solução mapeada é o projeto desenhar o próprio teclado e enviar as teclas com `pyautogui.press()`, mecanismo já testado e funcional. O detalhamento está na seção 7 do `docs/passo_a_passo.md`.